# Perbandingan Model — Prediksi Harga Saham Toyota

Notebook ini membandingkan **7 model** untuk prediksi harga saham Toyota (Close price):

| Model | Tipe |
|---|---|
| ARIMA | Statistical / Time Series |
| Prophet | Statistical / Time Series |
| LSTM | Deep Learning (PyTorch) |
| GRU | Deep Learning (PyTorch) |
| Random Forest | Machine Learning |
| XGBoost | Machine Learning |
| Linear Regression (Ridge) | Machine Learning |

**Metrik evaluasi:** MAE, RMSE, MAPE, Directional Accuracy, R²  
**Split:** 80% train / 20% test (urutan waktu, bukan random)

---
## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Statistical
from statsmodels.tsa.arima.model import ARIMA
from prophet import Prophet

# Deep Learning
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import MinMaxScaler

# ML
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Device (GPU jika tersedia)
try:
    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    if DEVICE.type == 'cuda':
        torch.zeros(1).to(DEVICE)   # test CUDA
except:
    DEVICE = torch.device('cpu')

print("Libraries loaded successfully ✓")
print(f"PyTorch: {torch.__version__} | Device: {DEVICE}")

---
## 2. Load Data & Define Split

In [ ]:
df = pd.read_csv('../Toyota_Stock_Prices_1980_2026.csv', parse_dates=['Date'])
df = df.sort_values('Date').reset_index(drop=True)
df = df[['Date', 'Close', 'Open', 'High', 'Low', 'Volume']].dropna()

# 80 / 20 split (temporal – no random shuffling)
split = int(len(df) * 0.8)
train_df = df.iloc[:split].copy()
test_df  = df.iloc[split:].copy()

close_series   = df['Close'].values.astype(np.float32)
train_close    = close_series[:split]
test_close     = close_series[split:]

print(f"Total rows : {len(df)}")
print(f"Train rows : {len(train_df)}  ({train_df['Date'].min().date()} → {train_df['Date'].max().date()})")
print(f"Test rows  : {len(test_df)}   ({test_df['Date'].min().date()} → {test_df['Date'].max().date()})")
print(f"Close range: ${close_series.min():.2f} – ${close_series.max():.2f}")

In [ ]:
def compute_metrics(actual, predicted, model_name=""):
    """Hitung MAE, RMSE, MAPE, Directional Accuracy, dan R²."""
    actual    = np.array(actual).flatten()
    predicted = np.array(predicted).flatten()
    mae     = mean_absolute_error(actual, predicted)
    rmse    = np.sqrt(mean_squared_error(actual, predicted))
    mape    = np.mean(np.abs((actual - predicted) / (actual + 1e-8))) * 100
    dir_acc = np.mean(np.sign(np.diff(actual)) == np.sign(np.diff(predicted))) * 100
    r2      = r2_score(actual, predicted)
    if model_name:
        print(f"{'='*50}")
        print(f"  {model_name}")
        print(f"{'='*50}")
        print(f"  MAE              : {mae:.4f}")
        print(f"  RMSE             : {rmse:.4f}")
        print(f"  MAPE             : {mape:.2f}%")
        print(f"  Directional Acc  : {dir_acc:.2f}%")
        print(f"  R²               : {r2:.4f}")
    return {"Model": model_name, "MAE": mae, "RMSE": rmse,
            "MAPE (%)": mape, "Dir Acc (%)": dir_acc, "R²": r2}

all_results = []   # kumpulkan semua hasil di sini
all_preds   = {}   # simpan (actual, pred) untuk plot
print("Helper functions ready ✓")

---
## 3. ARIMA

In [ ]:
import time

# ARIMA pakai order (5,1,0) — sama seperti notebook individual
# Rolling 1-step forecast agar prediksi konsisten dengan test set
print("Training ARIMA (rolling 1-step forecast)...")
t0 = time.time()

history          = list(train_close)
arima_predictions = []

# Rolling forecast pada sebagian kecil test set (200 step) agar cepat
# gunakan semua test jika data tidak terlalu besar
N_TEST_ARIMA = min(len(test_close), 500)   # cap 500 step agar tidak terlalu lama

for i in range(N_TEST_ARIMA):
    model_arima = ARIMA(history, order=(5, 1, 0))
    result      = model_arima.fit()
    yhat        = result.forecast(steps=1)[0]
    arima_predictions.append(yhat)
    history.append(test_close[i])

arima_actual = test_close[:N_TEST_ARIMA]
r = compute_metrics(arima_actual, arima_predictions, "ARIMA (5,1,0)")
r["Train Time (s)"] = round(time.time() - t0, 1)
all_results.append(r)
all_preds["ARIMA"] = (arima_actual, np.array(arima_predictions))
print(f"Done in {r['Train Time (s)']}s")

---
## 4. Prophet

In [ ]:
print("Training Prophet...")
t0 = time.time()

prophet_train = train_df[['Date', 'Close']].rename(columns={'Date': 'ds', 'Close': 'y'})
prophet_test  = test_df[['Date', 'Close']].rename(columns={'Date': 'ds', 'Close': 'y'})

prophet_model = Prophet(
    weekly_seasonality=True,
    yearly_seasonality=True,
    daily_seasonality=False,
    changepoint_prior_scale=0.05
)
prophet_model.fit(prophet_train)

future   = prophet_model.make_future_dataframe(periods=len(prophet_test) + 30, freq='D')
forecast = prophet_model.predict(future)

# Align prediksi dengan tanggal test
comp = prophet_test.merge(forecast[['ds', 'yhat']], on='ds', how='inner')
r = compute_metrics(comp['y'].values, comp['yhat'].values, "Prophet")
r["Train Time (s)"] = round(time.time() - t0, 1)
all_results.append(r)
all_preds["Prophet"] = (comp['y'].values, comp['yhat'].values)
print(f"Done in {r['Train Time (s)']}s")

---
## 5. LSTM (PyTorch)

In [ ]:
class LSTMModel(nn.Module):
    def __init__(self, input_size=1, hidden_size=64, n_layers=2, dropout=0.2):
        super().__init__()
        self.lstm   = nn.LSTM(input_size, hidden_size, n_layers,
                              batch_first=True, dropout=dropout)
        self.fc     = nn.Linear(hidden_size, 1)
    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :])

class GRUModel(nn.Module):
    def __init__(self, input_size=1, hidden_size=64, n_layers=2, dropout=0.2):
        super().__init__()
        self.gru    = nn.GRU(input_size, hidden_size, n_layers,
                             batch_first=True, dropout=dropout)
        self.fc     = nn.Linear(hidden_size, 1)
    def forward(self, x):
        out, _ = self.gru(x)
        return self.fc(out[:, -1, :])

def create_sequences(data, window=60):
    X, y = [], []
    for i in range(window, len(data)):
        X.append(data[i-window:i])
        y.append(data[i])
    return np.array(X, dtype=np.float32), np.array(y, dtype=np.float32)

def train_rnn_model(ModelClass, close_scaled, split_idx, window=60,
                    n_epochs=50, hidden=64, n_layers=2, batch_size=64):
    """Train a RNN (LSTM or GRU) and return (actual, predicted) in original scale."""
    train_data = close_scaled[:split_idx]
    # Include window-sized overlap for test sequence construction
    test_data  = close_scaled[split_idx - window:]
    
    X_tr, y_tr = create_sequences(train_data, window)
    X_te, y_te = create_sequences(test_data,  window)
    
    X_tr = X_tr.reshape(-1, window, 1)
    X_te = X_te.reshape(-1, window, 1)
    
    ds = TensorDataset(torch.from_numpy(X_tr), torch.from_numpy(y_tr).unsqueeze(1))
    dl = DataLoader(ds, batch_size=batch_size, shuffle=False)
    
    model = ModelClass(input_size=1, hidden_size=hidden, n_layers=n_layers).to(DEVICE)
    opt   = torch.optim.Adam(model.parameters(), lr=1e-3)
    crit  = nn.MSELoss()
    sch   = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, patience=5, factor=0.5)
    
    best_loss, best_w = float('inf'), None
    for epoch in range(n_epochs):
        model.train()
        ep_loss = 0.0
        for xb, yb in dl:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad()
            loss = crit(model(xb), yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            ep_loss += loss.item()
        ep_loss /= len(dl)
        sch.step(ep_loss)
        if ep_loss < best_loss:
            best_loss = ep_loss
            best_w = {k: v.clone() for k, v in model.state_dict().items()}
    
    model.load_state_dict(best_w)
    model.eval()
    with torch.no_grad():
        pred_s = model(torch.from_numpy(X_te).to(DEVICE)).cpu().numpy()
    
    return y_te, pred_s.flatten()   # still in scaled space, will inverse below

print("RNN model classes & helpers ready ✓")

In [ ]:
print("Training LSTM...")
t0 = time.time()

# Scale only on training data
scaler_lstm = MinMaxScaler()
close_scaled_lstm = scaler_lstm.fit_transform(
    close_series.reshape(-1, 1)).flatten().astype(np.float32)

WINDOW = 60
y_te_sc, pred_sc = train_rnn_model(
    LSTMModel, close_scaled_lstm, split,
    window=WINDOW, n_epochs=50, hidden=64, n_layers=2
)
# Inverse scale
actual_lstm = scaler_lstm.inverse_transform(y_te_sc.reshape(-1,1)).flatten()
pred_lstm   = scaler_lstm.inverse_transform(pred_sc.reshape(-1,1)).flatten()

r = compute_metrics(actual_lstm, pred_lstm, "LSTM")
r["Train Time (s)"] = round(time.time() - t0, 1)
all_results.append(r)
all_preds["LSTM"] = (actual_lstm, pred_lstm)
print(f"Done in {r['Train Time (s)']}s")

---
## 6. GRU (PyTorch)

In [ ]:
print("Training GRU...")
t0 = time.time()

scaler_gru = MinMaxScaler()
close_scaled_gru = scaler_gru.fit_transform(
    close_series.reshape(-1, 1)).flatten().astype(np.float32)

y_te_sc_g, pred_sc_g = train_rnn_model(
    GRUModel, close_scaled_gru, split,
    window=WINDOW, n_epochs=50, hidden=64, n_layers=2
)
actual_gru = scaler_gru.inverse_transform(y_te_sc_g.reshape(-1,1)).flatten()
pred_gru   = scaler_gru.inverse_transform(pred_sc_g.reshape(-1,1)).flatten()

r = compute_metrics(actual_gru, pred_gru, "GRU")
r["Train Time (s)"] = round(time.time() - t0, 1)
all_results.append(r)
all_preds["GRU"] = (actual_gru, pred_gru)
print(f"Done in {r['Train Time (s)']}s")

---
## 7. Feature Engineering (ML Models)

ML models (Random Forest, XGBoost, Ridge) tidak bisa langsung pakai time-series mentah.  
Kita buat **lag features** dari Close price agar bisa dilatih sebagai supervised learning.

In [ ]:
def build_lag_features(df_in, lags=30):
    """Buat lag features dari Close price."""
    d = df_in[['Date', 'Close', 'Open', 'High', 'Low', 'Volume']].copy()
    for i in range(1, lags + 1):
        d[f'close_lag_{i}'] = d['Close'].shift(i)
    # Rolling features
    d['ma_5']   = d['Close'].shift(1).rolling(5).mean()
    d['ma_20']  = d['Close'].shift(1).rolling(20).mean()
    d['std_5']  = d['Close'].shift(1).rolling(5).std()
    d['return'] = d['Close'].pct_change().shift(1)
    d = d.dropna().reset_index(drop=True)
    target      = d['Close'].values
    feature_cols = [c for c in d.columns if c not in ['Date', 'Close']]
    features     = d[feature_cols].values
    return features, target, feature_cols

LAGS = 30
feat_all, tgt_all, feat_names = build_lag_features(df, lags=LAGS)

# Date-ordered split based on split index
# After computing lag features some rows are dropped; recalculate split
n_feat = len(feat_all)
ml_split = int(n_feat * 0.8)

X_ml_train, y_ml_train = feat_all[:ml_split], tgt_all[:ml_split]
X_ml_test,  y_ml_test  = feat_all[ml_split:],  tgt_all[ml_split:]

print(f"ML Feature matrix: {feat_all.shape}")
print(f"ML Train: {X_ml_train.shape[0]} | Test: {X_ml_test.shape[0]}")
print(f"Features ({len(feat_names)}): {feat_names[:5]} ...")

---
## 8. Random Forest

In [ ]:
print("Training Random Forest...")
t0 = time.time()

rf_model = RandomForestRegressor(
    n_estimators=200, max_depth=10, n_jobs=-1, random_state=42
)
rf_model.fit(X_ml_train, y_ml_train)
pred_rf = rf_model.predict(X_ml_test)

r = compute_metrics(y_ml_test, pred_rf, "Random Forest")
r["Train Time (s)"] = round(time.time() - t0, 1)
all_results.append(r)
all_preds["Random Forest"] = (y_ml_test, pred_rf)
print(f"Done in {r['Train Time (s)']}s")

---
## 9. XGBoost

In [ ]:
print("Training XGBoost...")
t0 = time.time()

xgb_model = XGBRegressor(
    n_estimators=500, learning_rate=0.05, max_depth=5,
    subsample=0.8, colsample_bytree=0.8,
    objective='reg:squarederror',
    random_state=42, n_jobs=-1, verbosity=0
)
xgb_model.fit(
    X_ml_train, y_ml_train,
    eval_set=[(X_ml_test, y_ml_test)],
    verbose=False
)
pred_xgb = xgb_model.predict(X_ml_test)

r = compute_metrics(y_ml_test, pred_xgb, "XGBoost")
r["Train Time (s)"] = round(time.time() - t0, 1)
all_results.append(r)
all_preds["XGBoost"] = (y_ml_test, pred_xgb)
print(f"Done in {r['Train Time (s)']}s")

---
## 10. Linear Regression (Ridge)

In [ ]:
print("Training Ridge Regression...")
t0 = time.time()

ridge_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('model',  Ridge(alpha=1.0))
])
ridge_pipe.fit(X_ml_train, y_ml_train)
pred_ridge = ridge_pipe.predict(X_ml_test)

r = compute_metrics(y_ml_test, pred_ridge, "Ridge Regression")
r["Train Time (s)"] = round(time.time() - t0, 1)
all_results.append(r)
all_preds["Ridge Regression"] = (y_ml_test, pred_ridge)
print(f"Done in {r['Train Time (s)']}s")

---
## 11. Tabel Perbandingan Semua Model

In [ ]:
df_res = pd.DataFrame(all_results).set_index("Model")
metric_cols = ["MAE", "RMSE", "MAPE (%)", "Dir Acc (%)", "R²"]

def highlight_best(df):
    """Highlight best (lowest for error metrics, highest for accuracy/R²)."""
    styles = pd.DataFrame('', index=df.index, columns=df.columns)
    # Lower is better
    for col in ["MAE", "RMSE", "MAPE (%)"]:
        if col in df.columns:
            best_idx = df[col].idxmin()
            styles.loc[best_idx, col] = 'background-color: #b7f7c4; font-weight: bold'
    # Higher is better
    for col in ["Dir Acc (%)", "R²"]:
        if col in df.columns:
            best_idx = df[col].idxmax()
            styles.loc[best_idx, col] = 'background-color: #b7f7c4; font-weight: bold'
    return styles

styled = (
    df_res[metric_cols]
    .style.apply(highlight_best, axis=None)
    .format({"MAE": "{:.4f}", "RMSE": "{:.4f}",
             "MAPE (%)": "{:.2f}%", "Dir Acc (%)": "{:.2f}%", "R²": "{:.4f}"})
    .set_caption("Perbandingan Metrik Evaluasi — Toyota Stock Price Forecasting")
)
styled

---
## 12. Visualisasi Perbandingan

In [ ]:
colors7 = ['#e53935','#FB8C00','#8E24AA','#1E88E5','#43A047','#F4511E','#00ACC1']
models_list = df_res.index.tolist()

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# ── Bar: MAE & RMSE ──────────────────────────────────────
ax = axes[0]
x = np.arange(len(models_list)); w = 0.35
b1 = ax.bar(x - w/2, df_res['MAE'].values,  w, label='MAE',  color='steelblue',  alpha=0.85)
b2 = ax.bar(x + w/2, df_res['RMSE'].values, w, label='RMSE', color='darkorange', alpha=0.85)
ax.set_xticks(x); ax.set_xticklabels(models_list, rotation=30, ha='right', fontsize=9)
ax.set_title('MAE vs RMSE', fontweight='bold'); ax.legend(); ax.grid(axis='y', alpha=0.3)
for b in [*b1, *b2]:
    ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.1,
            f'{b.get_height():.2f}', ha='center', va='bottom', fontsize=7)

# ── Bar: MAPE ────────────────────────────────────────────
ax = axes[1]
bars = ax.bar(models_list, df_res['MAPE (%)'].values, color=colors7, alpha=0.85)
ax.set_xticklabels(models_list, rotation=30, ha='right', fontsize=9)
ax.set_title('MAPE (%)', fontweight='bold'); ax.grid(axis='y', alpha=0.3)
for b in bars:
    ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.05,
            f'{b.get_height():.2f}%', ha='center', va='bottom', fontsize=8)

# ── Bar: Directional Accuracy ────────────────────────────
ax = axes[2]
bars3 = ax.bar(models_list, df_res['Dir Acc (%)'].values, color=colors7, alpha=0.85)
ax.set_xticklabels(models_list, rotation=30, ha='right', fontsize=9)
ax.set_ylim(0, 110); ax.axhline(50, color='red', ls='--', lw=1, label='Random (50%)')
ax.set_title('Directional Accuracy (%)', fontweight='bold')
ax.legend(fontsize=9); ax.grid(axis='y', alpha=0.3)
for b in bars3:
    ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.5,
            f'{b.get_height():.1f}%', ha='center', va='bottom', fontsize=8)

plt.suptitle("Perbandingan Metrik — Toyota Stock Forecasting",
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Plot prediksi tiap model vs aktual (subset 300 titik agar tidak penuh)
n_show = 300
fig, axes = plt.subplots(4, 2, figsize=(18, 20))
axes = axes.flatten()

for idx, (name, color) in enumerate(zip(all_preds.keys(), colors7)):
    actual, pred = all_preds[name]
    n = min(n_show, len(actual))
    ax = axes[idx]
    ax.plot(actual[:n],   color='black',      lw=1.5, label='Aktual', alpha=0.8)
    ax.plot(pred[:n],     color=color,        lw=1.5, label=name,   alpha=0.9, linestyle='--')
    metrics_row = df_res.loc[name] if name in df_res.index else None
    title_suffix = ""
    if metrics_row is not None:
        title_suffix = f"  |  RMSE={metrics_row['RMSE']:.2f}  MAPE={metrics_row['MAPE (%)']:.1f}%"
    ax.set_title(f"{name}{title_suffix}", fontsize=10, fontweight='bold')
    ax.legend(fontsize=8, loc='upper left')
    ax.grid(alpha=0.3)
    ax.set_xlabel("Step (test period)")
    ax.set_ylabel("Close Price ($)")

# Hide last empty axes if any
for ax in axes[len(all_preds):]:
    ax.set_visible(False)

plt.suptitle("Prediksi vs Aktual — Toyota Stock (Test Set)",
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

---
## 13. Kesimpulan

In [ ]:
best_rmse   = df_res['RMSE'].idxmin()
best_mape   = df_res['MAPE (%)'].idxmin()
best_dir    = df_res['Dir Acc (%)'].idxmax()
best_r2     = df_res['R²'].idxmax()

# Ranking: normalisasi setiap metrik ke [0,1] lalu gabung
df_rank = df_res[metric_cols].copy()
# Invert error metrics (lower is better → higher rank)
for col in ['MAE', 'RMSE', 'MAPE (%)']:
    df_rank[col] = 1 - (df_rank[col] - df_rank[col].min()) / (df_rank[col].max() - df_rank[col].min() + 1e-9)
# Normalize higher-is-better metrics
for col in ['Dir Acc (%)', 'R²']:
    r_min, r_max = df_rank[col].min(), df_rank[col].max()
    df_rank[col] = (df_rank[col] - r_min) / (r_max - r_min + 1e-9)

df_rank['Composite Score'] = df_rank[metric_cols].mean(axis=1)
best_overall = df_rank['Composite Score'].idxmax()

print("=" * 62)
print("  KESIMPULAN PERBANDINGAN MODEL — TOYOTA STOCK")
print("=" * 62)
print(f"\n{'Model':<22} {'MAE':>8} {'RMSE':>8} {'MAPE%':>8} {'DirAcc%':>9} {'R²':>7}")
print("─" * 65)
for m in df_res.index:
    row = df_res.loc[m]
    print(f"{m:<22} {row['MAE']:>8.4f} {row['RMSE']:>8.4f} {row['MAPE (%)']:>7.2f}%"
          f" {row['Dir Acc (%)']:>8.2f}% {row['R²']:>7.4f}")
print("─" * 65)

print(f"\n🏆 Terbaik per metrik:")
print(f"   Lowest MAE     → {best_rmse:<22} ({df_res.loc[best_rmse,'MAE']:.4f})")
print(f"   Lowest RMSE    → {best_rmse:<22} ({df_res.loc[best_rmse,'RMSE']:.4f})")
print(f"   Lowest MAPE    → {best_mape:<22} ({df_res.loc[best_mape,'MAPE (%)']:.2f}%)")
print(f"   Best Dir Acc   → {best_dir:<22} ({df_res.loc[best_dir,'Dir Acc (%)']:.2f}%)")
print(f"   Best R²        → {best_r2:<22} ({df_res.loc[best_r2,'R²']:.4f})")

print(f"\n🥇 Model TERBAIK KESELURUHAN (Composite Score):")
print(f"   ➜  {best_overall}")

medals = ["🥇","🥈","🥉","  4.","  5.","  6.","  7."]
ranking = df_rank['Composite Score'].sort_values(ascending=False)
print("\n📝 Ranking semua model:")
for i, (name, score) in enumerate(ranking.items()):
    print(f"   {medals[i]} {name:<22} → score {score:.4f}")

print("\n📌 Catatan:")
print("   • ML models (RF, XGBoost, Ridge) menggunakan lag features — hasil bisa sangat baik")
print("     karena memiliki akses langsung ke harga hari sebelumnya.")
print("   • ARIMA & Prophet dijalankan secara penuh sebagai time-series models.")
print("   • LSTM/GRU dilatih dgn sequence windows — bagus untuk pola non-linear.")